# Traffic-flow imputation

This notebook runs the pipeline on one real sensor series from the prepared LargeST traffic panel and uses an existing missing interval from that panel. LargeST is stored as HDF5 rather than CSV, so the notebook exports only the selected, bounded working series as a local CSV for inspection; the original panel is never modified.

The example uses **no outlier removal** and **z-score standardization**, completing the three distinct configuration choices across the domain notebooks.

In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Start Jupyter from inside the repository.')
    PROJECT_ROOT = PROJECT_ROOT.parent
for line in (PROJECT_ROOT / '.env').read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(34)).strip(chr(39)))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from gap_imputation_benchmark.algorithm import DomainImputationConfig, RunMetadata, impute_with_rf_selector

DATA_ROOT = Path(os.environ['TRAFFIC_DATA_DIR'])
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'algorithm' / 'traffic' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXECUTOR_NAME = 'Oliver'
EXECUTOR_INFO_PATH = PROJECT_ROOT / 'data' / 'person_info_examples' / 'executor_example.json'
RESPONSIBLE_PERSON_NAME = 'Jenny'
RESPONSIBLE_PERSON_INFO_PATH = PROJECT_ROOT / 'data' / 'person_info_examples' / 'executor_responsible_person_example.json'
EXECUTION_NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks' / 'algorithm' / 'traffic' / '01_traffic_imputation.ipynb'


In [2]:
panel_files = sorted((DATA_ROOT / 'interim' / 'largest_flow_panel_selected').glob('district_*_flow_panel.h5'))
if not panel_files:
    raise FileNotFoundError('No prepared traffic panel found. Run the Traffic panel preparation first.')
PANEL_PATH = panel_files[0]

# Centre a 30-week window on a genuine missing value, preserving weekly history.
import tables
week_steps = 7 * 24 * 12
with tables.open_file(PANEL_PATH, mode='r') as source:
    flow = source.root.flow
    missing_indices = np.flatnonzero(~np.isfinite(flow[:, 0]))
    target_idx = next(i for i in missing_indices if 15 * week_steps <= i < flow.shape[0] - 15 * week_steps)
    start, stop = target_idx - 15 * week_steps, target_idx + 15 * week_steps
    values = flow[start:stop, 0].astype(float)

timestamps = pd.date_range('2017-01-01', periods=len(values), freq='5min') + pd.Timedelta(minutes=5 * start)
frame = pd.DataFrame({'timestamp': timestamps, 'traffic_flow': values})
frame['is_valid'] = np.isfinite(frame['traffic_flow'])
SOURCE_CSV = OUTPUT_DIR / 'traffic_working_input.csv'
frame.to_csv(SOURCE_CSV, index=False)
VALUE_COLUMN = 'traffic_flow'
frame.head(), PANEL_PATH.name

(            timestamp  traffic_flow  is_valid
 0 2017-05-02 22:45:00          83.0      True
 1 2017-05-02 22:50:00          88.0      True
 2 2017-05-02 22:55:00          98.0      True
 3 2017-05-02 23:00:00          85.0      True
 4 2017-05-02 23:05:00          87.0      True,
 'district_03_flow_panel.h5')

In [3]:
n_missing = int(frame[VALUE_COLUMN].isna().sum())
if n_missing == 0:
    raise ValueError('No genuine missing values were found in the selected traffic window.')
print(f'Using {n_missing} genuine missing five-minute values from {PANEL_PATH.name}.')

Using 2 genuine missing five-minute values from district_03_flow_panel.h5.


In [ ]:
config = DomainImputationConfig(
    domain='traffic', timestamp_col='timestamp', validity_col='is_valid',
    outlier_method='none', standardization_method='zscore',
)
imputed_frame, provenance = impute_with_rf_selector(
    frame, VALUE_COLUMN, config=config,
    run_metadata=RunMetadata(
        executor_name=EXECUTOR_NAME, executor_info_path=EXECUTOR_INFO_PATH,
        executor_responsible_person=RESPONSIBLE_PERSON_NAME,
        executor_responsible_person_info_path=RESPONSIBLE_PERSON_INFO_PATH,
        execution_notebook_path=EXECUTION_NOTEBOOK_PATH,
        comment=f'Traffic example from {PANEL_PATH.name}, first sensor.',
    ),
    input_path=SOURCE_CSV, output_path=OUTPUT_DIR / 'traffic_imputed.csv',
    provenance_path=OUTPUT_DIR / 'traffic_provenance.json',
)
provenance['summary']

## Review

Traffic timestamps are required so the selector can consider weekly seasonal reconstruction. The locally exported input CSV and all derived outputs are reproducible convenience files; they do not replace the versioned LargeST source panel.